# Hole-Cleaning Conditions — End-to-End Walkthrough

This notebook narrates the full workflow using the `holecleaning` package: from raw
experimental data through exploratory analysis, the density and fluid-velocity studies,
model benchmarking, drilling-parameter optimisation, and the hole-cleaning **zone / feed-zone**
analysis.

Each stage here mirrors a script in `scripts/`; the package does the heavy lifting so the
notebook stays readable. Run top to bottom to reproduce every figure and table.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '../src')   # so `import holecleaning` works from notebooks/

from holecleaning import config as C
from holecleaning import data, features, eda, models, evaluation, optimization, zones

evaluation.apply_style()
print('Project root:', C.ROOT_DIR)

## 1. Load & clean the data

We drop the row index (`Test No.`) and the redundant, perfectly-collinear plastic-viscosity
column (`PV`), keeping `YP` as the rheology representative. The target is the cuttings-bed
volume fraction, `Concentration`.

In [ ]:
df = data.load_clean()
print(df.shape)
data.describe(df)

## 2. Exploratory analysis

Correlation structure, distributions, and a dual feature ranking (Pearson vs. Random-Forest
Gini). Agreement between the two methods raises confidence in the identified drivers.

In [ ]:
eda.correlation_matrix(df)
eda.feature_distributions(df)
eda.target_relationships(df)
ranking = eda.feature_ranking(df)
ranking

## 3. Density & fluid-velocity analysis

The two dominant physical levers. Annular velocity is derived from flow rate and the
(constant) flow-loop geometry defined in `config.py`.

In [ ]:
print('Density vs. mean concentration')
display(features.density_table(df))
print('\nAnnular velocity vs. mean concentration')
features.velocity_table(df)

In [ ]:
# Physics-informed features used downstream
featured = features.add_physics_features(df)
featured[['Flow rate','Annular velocity','Transport index','Carrying capacity index']].head()

## 4. Benchmark the model zoo

Ten regressors across three families are compared on identical 5-fold splits, plus a
stacked ensemble. Lower cross-validated RMSE is better.

In [ ]:
zoo = models.model_zoo()
zoo['Stacked'] = models.build_stack()
bench = evaluation.benchmark(zoo, *data.split(df)[::2])   # X_train, y_train
bench

In [ ]:
X_train, X_test, y_train, y_test = data.split(df)
bench = evaluation.benchmark(zoo, X_train, y_train)
evaluation.plot_benchmark(bench)
bench

## 5. Train, evaluate & explain the champion

We fit the Random Forest, inspect parity/residuals, and attribute its predictions with
model-agnostic permutation importance.

In [ ]:
champion = models.model_zoo()['RandomForest']
report = evaluation.evaluate_fitted(champion, X_train, y_train, X_test, y_test)
report

In [ ]:
tr_pred = champion.predict(X_train)
te_pred = champion.predict(X_test)
evaluation.plot_parity(y_train, tr_pred, y_test, te_pred, model_name='Random Forest')
evaluation.plot_residuals(y_test, te_pred, model_name='Random Forest')
_ = evaluation.plot_learning_curve(champion, df.drop(columns=[C.TARGET]), df[C.TARGET], model_name='Random Forest')

In [ ]:
_, importance = evaluation.plot_permutation_importance(
    champion, df.drop(columns=[C.TARGET]), df[C.TARGET], model_name='Random Forest')
importance

## 6. Drilling-parameter optimisation

For a single run, search a ±20% neighbourhood of the controllable parameters
(flow rate, pipe rotation, ROP) for the setting that minimises predicted concentration.

In [ ]:
champion.fit(X_train, y_train)
current = df.drop(columns=[C.TARGET]).iloc[1]
best_settings, best_conc = optimization.optimize_row(champion, current, pct_range=0.2)
print('Optimised concentration:', best_conc)
best_settings

## 7. Hole-cleaning zones & feed-zone identification

Bin the predictions into Efficient / Marginal / Poor zones and sweep the model across the
operating envelope. The red **feed zones** are where a stationary cuttings bed accumulates.

In [ ]:
champion.fit(df.drop(columns=[C.TARGET]), df[C.TARGET])
print(zones.zone_summary(df))

grid, XX, YY, ZZ = zones.build_zone_map(champion, df, 'Flow rate', 'Inclination')
zones.plot_zone_map(XX, YY, ZZ, 'Flow rate', 'Inclination',
                    model_name='Random Forest', overlay_df=df, filename=None)
zones.extract_feed_zones(grid, 'Flow rate', 'Inclination')

---
**Takeaways.** Annular velocity, pipe rotation and mud density dominate hole cleaning.
Raising annular velocity above ~1.2 ft/s moves operations out of the poor-cleaning feed
zone, and heavier mud provides additional carrying capacity. See `README.md` for the full
discussion and `reports/` for all generated artefacts.